# V53 - Margin-gated override on the frozen champion

Pre-registered hypothesis: ranking errors concentrate where the leader is unsure, measured by a razor-thin top-1 vs top-2 margin (champion minimum is 0.0002). Users whose champion margin is below 0.05 get the assessment-view ranking instead. No-chat users fall under this rule automatically, since an empty text row cannot produce a confident margin. Validation replays the same rule on OOF chat predictions under a fresh seed, using the OOF margin lower quartile as the threshold analogue. Three locks: replay slice CI lower above +0.01, global worst-fold non-negative, test geometry confined to sub-0.05-margin users.

In [1]:
import json
import re
from hashlib import sha256
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 4242
N_SPLITS = 5
SLICE_FLOOR = 0.01
CHAMP_MARGIN_T = 0.05
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
CHAMPION = next((p for p in [Path("/kaggle/input/intelligo-champion/submission_v42_primary.csv"),
                            Path("results/v42/submission_v42_primary.csv")] if p.exists()), None)
assert CHAMPION is not None, "v53 needs the champion file as its base"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v53")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve(), "| OUT:", OUT.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data | OUT: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/results/v53


## Views, margins, and replay rule

Chat and assessment Ridge under fresh seed 4242. The replay rule overrides OOF chat rows whose own margin falls in the bottom quartile, mirroring the champion-margin rule without touching labels.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)

by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
all_users = list(train["user_id"]) + test_ids

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return np.nan

A = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
A = np.where(np.isnan(A), np.nanmean(A, axis=0), A)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
T = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=3).fit_transform(docs)
A_tr, A_te = A[:n_train], A[n_train:]
T_tr, T_te = T[:n_train].toarray(), T[n_train:].toarray()

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def margin(S):
    s = np.sort(S, axis=1)[:, ::-1]
    return s[:, 0] - s[:, 1]

def ridge_oof(Xtr, Xte, alpha=3.0):
    pred = np.zeros((Xte.shape[0], len(MODS)))
    oof = np.zeros_like(Y)
    fold_id = np.zeros(len(Xtr), int)
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fi, (tr, va) in enumerate(skf.split(Xtr, strata, fold_groups)):
        for j in range(len(MODS)):
            m = Ridge(alpha=alpha, random_state=SEED)
            m.fit(Xtr[tr], Y[tr, j])
            oof[va, j] = m.predict(Xtr[va])
            pred[:, j] += m.predict(Xte) / N_SPLITS
        fold_id[va] = fi + 1
    return oof, pred, fold_id

oof_c, pred_c, fid = ridge_oof(T_tr, T_te)
oof_a, pred_a, _ = ridge_oof(A_tr, A_te)
print(f"chat OOF(seed4242): {ndcg5(Y, oof_c).mean():.5f} | assess OOF: {ndcg5(Y, oof_a).mean():.5f}")
q1 = np.quantile(margin(oof_c), 0.25)
print(f"OOF chat margin Q1={q1:.4f} (replay threshold analogue)")

chat OOF(seed4242): 0.62132 | assess OOF: 0.42087
OOF chat margin Q1=0.0477 (replay threshold analogue)


## Replay gate, then test geometry

The replay applies the margin rule to OOF predictions. Only if the replay slice clears the floor with no negative fold does the champion-margin rule touch the test file, and only inside the sub-threshold set.

In [3]:
base = ndcg5(Y, oof_c)
replay_mask = margin(oof_c) < np.quantile(margin(oof_c), 0.25)
act = oof_c.copy()
act[replay_mask] = oof_a[replay_mask]
d = ndcg5(Y, act) - base
ds = d[replay_mask]
lo_s, hi_s = np.quantile([rng.choice(ds, len(ds), replace=True).mean() for _ in range(5000)], [0.025, 0.975])
wf = min(ndcg5(Y[fid == k], act[fid == k]).mean() - base[fid == k].mean() for k in range(1, N_SPLITS + 1))
print(f"replay users={replay_mask.sum()} pooled={d.mean():+.5f} slice={ds.mean():+.5f} CI95=[{lo_s:+.5f},{hi_s:+.5f}] worst-fold={wf:+.5f}")

ch = pd.read_csv(CHAMPION)
assert ch["user_id"].tolist() == test_ids
C = ch[MODS].to_numpy(float)
test_mask = margin(C) < CHAMP_MARGIN_T
S = C.copy()
S[test_mask] = pred_a[test_mask]
changed = set(np.where((S != C).any(1))[0].tolist())
allowed = set(np.where(test_mask)[0].tolist())
print(f"test sub-threshold users={test_mask.sum()} changed={len(changed)} confined={changed <= allowed} top1-changes={(C.argmax(1) != S.argmax(1)).sum()}")

adopt = (lo_s > SLICE_FLOOR) and (wf >= 0) and (changed <= allowed)
print("ADOPT ACTION" if adopt else "FAIL-CLOSE: champion untouched")
final = S if adopt else C
sub = pd.DataFrame(final, columns=MODS)
sub.insert(0, "user_id", test_ids)
sub.to_csv(OUT / "submission_v53_primary.csv", index=False)
print("wrote", OUT / "submission_v53_primary.csv", "sha", sha256(pd.util.hash_pandas_object(sub).values.tobytes()).hexdigest()[:16])

replay users=1000 pooled=-0.03892 slice=-0.15569 CI95=[-0.17116,-0.14071] worst-fold=-0.04811
test sub-threshold users=194 changed=194 confined=True top1-changes=165
FAIL-CLOSE: champion untouched
wrote results/v53/submission_v53_primary.csv sha 7bc841d13df9f947
